In [44]:
# import necessary packages
import cmapPy.pandasGEXpress.GCToo as GCToo
from cmapPy.pandasGEXpress.parse import parse
import cmapPy.pandasGEXpress.subset_gctoo as sg

# import necessary preprocessing and data visualization tools
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# import modeling tools
import statsmodels.api as sm
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression

## Functions to Automate Generating Datasets

In [45]:
### function to select the dose with the highest tas score for samples treated by the same cell line and compound
def select_dose(cell_line_sigs):
    ## initialize an empty dataframe
    one_dose_sigs = pd.DataFrame()
    ## store all the unique compounds in the
    cps = cell_line_sigs['pert_iname'].unique()
    ## for each unique compound
    for cp in cps:
        # among the samples treated with that compound
        cp_sigs = cell_line_sigs[cell_line_sigs['pert_iname'] == cp]
        # add the sample with the highest tas score to the new dataframe
        one_dose_sigs = one_dose_sigs.append(cp_sigs.head(1))

    return one_dose_sigs

In [46]:
### function to select tas scores based on cell line at any tas cutoff
def select_samples(cell_line, tas_cutoff, sig_info):
    ## store all the drugs that have a tas score above the tas cutoff
    sigs_above_tas = sig_info[sig_info['tas'] > tas_cutoff]
    ## select all samples treated with the cell line of interest
    cell_line_sigs = sigs_above_tas[sigs_above_tas['cell_id'] == cell_line]
    ## sort the signatures from highest to lowest tas scores
    cell_line_sigs = cell_line_sigs.sort_values(by='tas', ascending=False)
    ## create a new dataframe containing the samples with the highest dose for each compound treated
    one_dose_sigs = select_dose(cell_line_sigs)
    
    return one_dose_sigs

In [47]:
### function to create a new dataframe containing compound-indication pairs, their tas scores, and spearman correlations
def spearman_corr(sig_info):
    ## 1. perform pairwise spearman correlations
    spearman_corrs = sig_info.data_df.corr(method='spearman')
    # store the sig_ids of all the drugs
    all_cps = spearman_corrs.index
    ##
    
    ## 2. create a new dataframe with all the spearman correlation information
    # create a list with the names of the columns for the dataframe
    columns = ['drug1', 'drug1 tas', 'drug2', 'drug2 tas', 'spearman correlation']
    # create an empty dataframe to store the relevant information
    spearman_corr_info = pd.DataFrame(columns=columns)
    
    # for each drug available,
    for sig_id1 in all_cps:
        # store the names and sig_ids of all the other drugs
        other_cps = all_cps[all_cps != sig_id1]
        # get the the spearman correlations with all the other drugs
        corrs = spearman_corrs.loc[other_cps, sig_id1]
        # get the name and tas the second drug
        cp2_info = sig_info.col_metadata_df.loc[other_cps, ['pert_iname', 'tas']]
        
        # create a dataframe containing the names of the other drugs, their tas, and spearman correlations
        cp_corr_info = pd.concat([cp2_info, corrs], axis=1).reset_index(drop=True)
        # rename the column names
        cp_corr_info.columns = ['drug2', 'drug2 tas', 'spearman correlation']
        # get the name of drug 1
        cp1_name = sig_info.col_metadata_df.loc[sig_id1, 'pert_iname']
        # add the name of drug 1
        cp_corr_info['drug1'] = [cp1_name]*len(cp_corr_info)
        # get the tas of drug1 
        cp1_tas = sig_info.col_metadata_df.loc[sig_id1, 'tas']
        # add the name of drug 1
        cp_corr_info['drug1 tas'] = [cp1_tas]*len(cp_corr_info)
        
        # add this data onto the running dataframe for all spearman correlations for all drugs
        spearman_corr_info = spearman_corr_info.append(cp_corr_info, ignore_index=True, sort=True)
    ##
    
    return spearman_corr_info

In [48]:
### function to add a column determining whether drug2 can treat a known indication of drug1
def known_indication(row):
    ## get a list of indications of drug1
    drug1_indications = row['drug1 indication'].split('|')
    ## get the indication of drug2
    drug2_indication = row['drug2 indication']
    ## create a boolean whether drug2's indication is a known indication of drug1
    known_indication = drug2_indication in drug1_indications
    
    return known_indication

In [49]:
### function to add the indications of the drugs
def add_indications(spearman_corrs, drug_inds, drug_ind):
    ## add the indications for the drug 1 in the spearman correlation pair
    corrs_with_ind = pd.merge(spearman_corrs, drug_inds[['pert_iname', 'indication']], left_on='drug1', right_on='pert_iname', how='left')
    ## add the indications for each of the drug 2s in the spearman correlation pair
    corrs_with_ind = pd.merge(corrs_with_ind, drug_ind, left_on='drug2', right_on='pert_iname', how='left')
    
    ## remove the pert_iname columns
    corrs_with_ind.drop(columns=['pert_iname_x', 'pert_iname_y'], axis=1, inplace=True)
    ## rename the indication columns
    corrs_with_ind.rename(columns={'indication_x': 'drug1 indication', 'indication_y': 'drug2 indication'}, inplace=True)
    
    ## add a column determining if drug2 is a known indication of drug1
    corrs_with_ind['known indication'] = corrs_with_ind.apply(known_indication, axis=1)
    
    return corrs_with_ind

In [50]:
### function to remove duplicate drug-indication pairs
def remove_duplicates(corrs_with_ind):
    ## sort the drug-indication pairs from highest to lowest spearman correlation between the drugs
    by_drug2_tas = corrs_with_ind.sort_values(by='spearman correlation', ascending=False)
    
    ## for duplicate drug-indication pairs, keep only the one with the one with the highest correlation
    unique_drug_ind = by_drug2_tas.drop_duplicates(subset=['drug1', 'drug2 indication'], keep='first')
    
    return unique_drug_ind

In [51]:
### function to automate the process of determining the spearman correlations for a cell line
def corrs_by_cell_line(cell_line, tas_cutoff, sig_info, drug_inds, drug_ind):
    ### 1. select one drug per sample for that cell line with a TAS > 0.2 and known indications
    samples2compare = select_samples(cell_line, tas_cutoff, sig_info)
    ###

    ### 2. select the gene signatures
    # read the LINCS Level 5 dataset
    file_path = "~/LINCS/ref_data/LINCS_dataset.gctx"

    # select only the landmark gene of drug compounds of interest
    gene_sigs = parse(file_path, rid=lm_gene_id, cid=samples2compare.index)

    # create a new GCTOO object with all the metadata for pair 
    gene_sig_info = GCToo.GCToo(data_df=gene_sigs.data_df.copy(), 
                            row_metadata_df=gene_sigs.row_metadata_df.copy(), 
                            col_metadata_df=samples2compare,
                            make_multiindex=True)
    ###

    ### 3. perform a pairwise spearman correlation
    spearman_corrs = spearman_corr(gene_sig_info)
    ###
    
    ### 4. add the indications of the compound-indication pairs
    corrs_with_indications = add_indications(spearman_corrs, drug_inds, drug_ind)
    ###
    
    ### 5. remove duplicate drug-indication pairs and keep only the pair with the highest tas for drug 2
    unique_drug_indications = remove_duplicates(corrs_with_indications)
    ###
    
    return unique_drug_indications

## Generate Training Datasets

#### *1. Select only the 978 landmark genes and all the samples that had a high transcriptional response to drug compounds*

In [52]:
# read gene info from gene_info metadata
file_path = "~/LINCS/ref_data/GSE70138_Broad_LINCS_gene_info_2017-03-06.txt"
gene_info = pd.read_csv(file_path, sep="\t", dtype=str)
gene_info.head()

,pr_gene_id,pr_gene_symbol,pr_gene_title,pr_is_lm,pr_is_bing
0,780,DDR1,discoidin domain receptor tyrosine kinase 1,1,1
1,7849,PAX8,paired box 8,1,1
2,2978,GUCA1A,guanylate cyclase activator 1A,0,0
3,2049,EPHB3,EPH receptor B3,0,1
4,2101,ESRRA,estrogen related receptor alpha,0,1


In [53]:
# select the ids of only the landmark genes
lm_gene_id = gene_info["pr_gene_id"][gene_info["pr_is_lm"] == "1"]

# check that I have the 978 landmark genes
print(len(lm_gene_id)) 

978


#### *2. Import the known indications for the drugs used in the LINCS dataset (Drug Repurposing Hub)*

In [54]:
# import the collapsed version of the known drug indications
collapsed_known_ind = pd.read_csv('~/LINCS/ref_data/drug_ind/processed/collapsed_LINCS_drug_repo_not_in_clin.txt')
collapsed_known_ind.head()

,pert_iname,indication
0,L-citrulline,Hypertension|Erectile Dysfunction
1,SN-38,Colorectal Neoplasms
2,abacavir,HIV
3,abiraterone-acetate,Prostatic Neoplasms
4,acarbose,Diabetes Mellitus


In [55]:
# import the version of drug indications (one row for every indication of a drug)
known_ind = pd.read_csv('~/LINCS/ref_data/drug_ind/processed/LINCS_drug_repo_not_in_clin.txt')
known_ind.head()

,pert_iname,indication
0,abacavir,HIV
1,abiraterone-acetate,Prostatic Neoplasms
2,acarbose,Diabetes Mellitus
3,acebutolol,Hypertension
4,acexamic-acid,Wound Healing


#### *3. Select the gene signatures ids of compounds of interest*

In [56]:
# path to file
file_path = "~/LINCS/ref_data/GSE70138_Broad_LINCS_sig_info.txt"
# import relevant information from the file as a dataframe with index as the sample id (sig_id)
sig_info = pd.read_csv(file_path, sep="\t", dtype=str, index_col=0).drop(columns='distil_id')
# filter only the drug compounds
sig_info = sig_info[sig_info['pert_type'] == 'trt_cp']
# remove the pert_type column
sig_info.drop(columns='pert_type', inplace=True)
sig_info.head()

,pert_id,pert_iname,cell_id,pert_idose,pert_itime
sig_id,,,,,
LJP005_A375_24H:A07,BRD-K76908866,CP-724714,A375,10.0 um,24 h
LJP005_A375_24H:A08,BRD-K76908866,CP-724714,A375,3.33 um,24 h
LJP005_A375_24H:A09,BRD-K76908866,CP-724714,A375,1.11 um,24 h
LJP005_A375_24H:A10,BRD-K76908866,CP-724714,A375,0.37 um,24 h
LJP005_A375_24H:A11,BRD-K76908866,CP-724714,A375,0.12 um,24 h


#### *4. Add the TAS scores for the samples*

In [57]:
# path to file
file_path = "~/LINCS/ref_data/GSE70138_Broad_LINCS_sig_metrics_2017-03-06.txt"
# import the tas scores from the file as a dataframe with sample id (sig_id) as the index 
tas_scores = pd.read_csv(file_path, usecols=['sig_id', 'tas'], sep='\t', index_col=0)
# remove all samples with a tas score < 0.2
tas_scores = tas_scores[tas_scores['tas'] > 0.2]

high_tas_sigs = pd.concat([sig_info, tas_scores], axis=1, sort=True)
high_tas_sigs = high_tas_sigs[high_tas_sigs['tas'].notna()]
high_tas_sigs.head()

,pert_id,pert_iname,cell_id,pert_idose,pert_itime,tas
LJP005_A375_24H:A07,BRD-K76908866,CP-724714,A375,10.0 um,24 h,0.442109
LJP005_A375_24H:A08,BRD-K76908866,CP-724714,A375,3.33 um,24 h,0.458614
LJP005_A375_24H:A10,BRD-K76908866,CP-724714,A375,0.37 um,24 h,0.229162
LJP005_A375_24H:A13,BRD-K85606544,neratinib,A375,10.0 um,24 h,0.397924
LJP005_A375_24H:A14,BRD-K85606544,neratinib,A375,3.33 um,24 h,0.449551


In [58]:
# get a list of all compounds available in LINCS dataset
LINCS_cps = high_tas_sigs['pert_iname'].unique()
# get a list of all compounds available in drug repurposing hub
known_cps = collapsed_known_ind['pert_iname'].to_list()
# get all the compounds in LINCS dataset that have a known indication
cps_with_indications = set(known_cps) & set(LINCS_cps)
# filter only samples that have been treated with compounds with known indications
LINCS_sig_info = high_tas_sigs[high_tas_sigs['pert_iname'].isin(cps_with_indications)]
LINCS_sig_info.head()

,pert_id,pert_iname,cell_id,pert_idose,pert_itime,tas
LJP005_A375_24H:A13,BRD-K85606544,neratinib,A375,10.0 um,24 h,0.397924
LJP005_A375_24H:A14,BRD-K85606544,neratinib,A375,3.33 um,24 h,0.449551
LJP005_A375_24H:A15,BRD-K85606544,neratinib,A375,1.11 um,24 h,0.421605
LJP005_A375_24H:A16,BRD-K85606544,neratinib,A375,0.37 um,24 h,0.385313
LJP005_A375_24H:A17,BRD-K85606544,neratinib,A375,0.12 um,24 h,0.342939


In [59]:
LINCS_sig_info.shape

(7862, 6)

#### *5. Generate the training datasets for drugs in clinical dataset*

In [60]:
### initialize a dictionary to store the datasets for each cell line
train_data = {}

### store a list of all the cell lines
cell_lines = ['MCF7', 'A375', 'PC3']

### store tas cutoff to use to perform analysis
tas_cutoff = 0.2

### for each cell line
for cell_line in cell_lines:
    ## get the summarized table containing all spearman correlations for each cell line
    spearman_corrs = corrs_by_cell_line(cell_line, tas_cutoff, LINCS_sig_info, collapsed_known_ind, known_ind)
    ## store this dataframe as text file in the training data folder
    file_path = '~/LINCS/ref_data/training_data/known_ind_only/' + cell_line +'_known_spearman_corrs.txt'
    #spearman_corrs.to_csv(file_path, index=False)
    ## store this dataset in the dictionary with the cell line as the key
    train_data[cell_line] = spearman_corrs

train_data['MCF7'].head()

,drug1,drug1 tas,drug2,drug2 tas,spearman correlation,drug1 indication,drug2 indication,known indication
81189,ixazomib-citrate,0.747696,carfilzomib,0.763093,0.839315,Multiple Myeloma,Multiple Myeloma,True
40581,carfilzomib,0.763093,ixazomib-citrate,0.747696,0.839315,Multiple Myeloma,Multiple Myeloma,True
41698,penfluridol,0.659227,lomitapide,0.706904,0.801042,Schizophrenia,Hypercholesterolemia,False
121977,lomitapide,0.706904,penfluridol,0.659227,0.801042,Hypercholesterolemia,Schizophrenia,False
37966,norgestrel,0.503617,fluocinolone-acetonide,0.518611,0.786903,Contraceptive Agents,"Dermatitis, Seborrheic",False


## Generate External Validation Datasets

#### *1. Import the clinical indications*

In [61]:
# import all the indications associated with a drug in the LINCS dataset
collapsed_clin_inds = pd.read_csv('~/LINCS/ref_data/drug_ind/processed/collapsed_LINCS_clinical_ind (AACT).txt')
collapsed_clin_inds.head()

,pert_iname,indication
0,2-methoxyestradiol,"Multiple Myeloma|Neoplasms, Plasma Cell|Gliobl..."
1,"3,3'-diindolylmethane",Breast Neoplasms|Prostatic Neoplasms|Thyroid D...
2,5-aminolevulinic-acid,"Bowen's Disease|Keratosis, Actinic|Keratosis|B..."
3,ACY-1215,"Multiple Myeloma|Neoplasms, Plasma Cell|Fallop..."
4,ASA-404,Neoplasms|Lung Neoplasms|Small Cell Lung Carci...


In [62]:
# import the clinical indications of drugs used in the LINCS dataset 
clin_ind = pd.read_csv('~/LINCS/ref_data/drug_ind/processed/LINCS_clinical_ind (AACT).txt', usecols=['pert_iname', 'indication'])
clin_ind.head()

,pert_iname,indication
0,2-methoxyestradiol,Multiple Myeloma
1,2-methoxyestradiol,"Neoplasms, Plasma Cell"
2,2-methoxyestradiol,Glioblastoma
3,2-methoxyestradiol,Carcinoid Tumor
4,2-methoxyestradiol,Prostatic Neoplasms


In [63]:
# get a list of all compounds available from the clinical dataset
clinical_cps = collapsed_clin_inds['pert_iname']
# get all the compounds in LINCS dataset that have a known indication
cps_with_indications = set(clinical_cps) & set(LINCS_cps)
# filter only samples that have been treated with compounds with known indications
clin_sig_info = high_tas_sigs[high_tas_sigs['pert_iname'].isin(cps_with_indications)]
clin_sig_info.head()

,pert_id,pert_iname,cell_id,pert_idose,pert_itime,tas
LJP005_A375_24H:A19,BRD-K78431006,crizotinib,A375,10.0 um,24 h,0.409499
LJP005_A375_24H:A20,BRD-K78431006,crizotinib,A375,3.33 um,24 h,0.371808
LJP005_A375_24H:A21,BRD-K78431006,crizotinib,A375,1.11 um,24 h,0.275981
LJP005_A375_24H:A23,BRD-K78431006,crizotinib,A375,0.12 um,24 h,0.232353
LJP005_A375_24H:C13,BRD-K56343971,vemurafenib,A375,10.0 um,24 h,0.639785


In [64]:
# get a list of all compounds available from the clinical dataset
clinical_cps = collapsed_clin_inds['pert_iname']
# get all the compounds in LINCS dataset that have a known indication
cps_with_indications = set(clinical_cps) & set(LINCS_cps)
# filter only samples that have been treated with compounds with known indications
clin_sig_info = high_tas_sigs[high_tas_sigs['pert_iname'].isin(cps_with_indications)]
clin_sig_info.head()

,pert_id,pert_iname,cell_id,pert_idose,pert_itime,tas
LJP005_A375_24H:A19,BRD-K78431006,crizotinib,A375,10.0 um,24 h,0.409499
LJP005_A375_24H:A20,BRD-K78431006,crizotinib,A375,3.33 um,24 h,0.371808
LJP005_A375_24H:A21,BRD-K78431006,crizotinib,A375,1.11 um,24 h,0.275981
LJP005_A375_24H:A23,BRD-K78431006,crizotinib,A375,0.12 um,24 h,0.232353
LJP005_A375_24H:C13,BRD-K56343971,vemurafenib,A375,10.0 um,24 h,0.639785


#### *2. Generate the validation datasets for drugs in clinical dataset*

In [66]:
### initialize a dictionary to store the datasets for each cell line
val_data = {}

### for each cell line
for cell_line in cell_lines:
    ## get the summarized table containing all spearman correlations for each cell line
    spearman_corrs = corrs_by_cell_line(cell_line, tas_cutoff, clin_sig_info, collapsed_clin_inds, clin_ind)
    ## store this dataframe as text file in the validation data folder
    #file_path = '~/LINCS/ref_data/val_data/' + cell_line +'_spearman_corrs.txt'
    #spearman_corrs.to_csv(file_path, index=False)
    ## store this dataset in the dictionary with the cell line as the key
    val_data[cell_line] = spearman_corrs

val_data['MCF7'].head()

,drug1,drug1 tas,drug2,drug2 tas,spearman correlation,drug1 indication,drug2 indication,known indication
4437903,lonidamine,0.763669,ixazomib,0.791817,0.897118,Prostatic Hyperplasia|Hyperplasia,"Lymphoma, Mantle-Cell",False
4437883,lonidamine,0.763669,ixazomib,0.791817,0.897118,Prostatic Hyperplasia|Hyperplasia,Lupus Nephritis,False
4437885,lonidamine,0.763669,ixazomib,0.791817,0.897118,Prostatic Hyperplasia|Hyperplasia,Graft vs Host Disease,False
4437886,lonidamine,0.763669,ixazomib,0.791817,0.897118,Prostatic Hyperplasia|Hyperplasia,Recurrence,False
4437887,lonidamine,0.763669,ixazomib,0.791817,0.897118,Prostatic Hyperplasia|Hyperplasia,Kidney Diseases,False
